In [ ]:
import os
from torch.utils.data import ConcatDataset, random_split, DataLoader
import data_utils

# ------------------
# Load Group 1a Data
# ------------------
data_path = '/path/to/plasma_sim_data/Group1a/'
h5_files = [os.path.join(data_path, i) for i in os.listdir(data_path)]
cache_dir = '/path/to/cache/group1a_preprocessed_cache'

G1a_dataset = data_utils.PlasmaSimDataset(
    hdf5_paths=h5_files,
    return_raw_data=False,
    domain_bounds=0.1,
    resolution=128,
    num_workers=64,
    cache_dir=cache_dir,
    data_transform=None,         # No per-video normalization here.
    target_transform=None        # Assuming you want raw video values.
)

filtered_G1a_dataset = data_utils.filter_dataset_by_frames(G1a_dataset, min_frames=10)

# ------------------
# Load Group 1b Data
# ------------------
data_path = '/path/to/plasma_sim_data/Group1b/'
h5_files = [os.path.join(data_path, i) for i in os.listdir(data_path)]
cache_dir = '/path/to/cache/group1b_preprocessed_cache'

G1b_dataset = data_utils.PlasmaSimDataset(
    hdf5_paths=h5_files,
    return_raw_data=False,
    domain_bounds=0.1,
    resolution=128,
    num_workers=64,
    cache_dir=cache_dir,
    data_transform=None,         # No per-video normalization here.
    target_transform=None        # Assuming you want raw video values.
)

filtered_G1b_dataset = data_utils.filter_dataset_by_frames(G1b_dataset, min_frames=10)

del G1a_dataset; del G1b_dataset

# -------------------------------
# Combine the two datasets
# -------------------------------
full_dataset = ConcatDataset([filtered_G1a_dataset, filtered_G1b_dataset])
print("Combined dataset size:", len(full_dataset))

del filtered_G1a_dataset; del filtered_G1b_dataset

# -------------------------------------
# Split into training and validation sets
# -------------------------------------
train_size = int(0.7 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
print("Train size:", len(train_dataset), "Val size:", len(val_dataset))

# ---------------------------------------------------------
# Compute global video stats on the training set only.
# ---------------------------------------------------------
global_mean, global_std = data_utils.compute_global_video_stats(
    train_dataset, transform=data_utils.LogTransform()
)
print("Global video mean:", global_mean, "Global video std:", global_std)

# ---------------------------------------------------------
# Compute normalization parameters for the targets.
# Note: Here, we assume you want to compute these from the filtered dataset.
# You may choose to compute them from the training set only if desired.
# ---------------------------------------------------------
# For example, if you have a function that returns:
#    property_norm_params, laser_norm_params = compute_normalization_params(dataset)
property_norm_params, laser_norm_params = data_utils.compute_normalization_params(train_dataset)
print("Target normalization parameters:")
print("Property:", property_norm_params)
print("Laser:", laser_norm_params)

# ---------------------------------------------------------
# Create a video transform that standardizes using the training-set stats.
# ---------------------------------------------------------
video_transform = data_utils.StandardizeVideo(global_mean,
                                              global_std,
                                              apply_log=True,
                                              target_size=(96,96))

target_transform = data_utils.ElementPropertyTransform( 
                 property_norm_params=property_norm_params,  # dict mapping each property to (min, max)
                 laser_norm_params=laser_norm_params      # dict mapping 'rspot' and 'laser_power_wcm' to (min, max)
                )

#update the data transforms
for subset in train_dataset.dataset.datasets:
    subset.dataset.data_transform = video_transform
    subset.dataset.target_transform = target_transform

for subset in val_dataset.dataset.datasets:
    subset.dataset.data_transform = video_transform
    subset.dataset.target_transform = target_transform

# Now create DataLoader instances.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

import models

model_config = {
    # spatiotemporal CNN encoder 
    'input_channels':1,
    'cnn_output_shape':1024,
    'num_cnn_layers':5,
    'blocks_per_stage':2,
    'start_filters':32,
    'input_frames':10,
    
    # mlp parameter encoder
    'param_input_shape':2,
    'param_hidden_layers':[8,8],
    'param_output_shape':8,
    
    # mlp prediction head
    'hidden_layers':[256,32],
    'output_shape':7
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.PropertyPredictor(model_config).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total number of parameters: {total_params}')

num_epochs = 200
optimizer = optim.Adam(model.parameters(), lr=0.001)  # You can adjust the learning rate
criterion = nn.MSELoss(reduction='mean')

checkpoint_filename = 'CNN ElementProperty Regressor Group1a-b.pth'

history = models.train_regressor(model, train_loader, val_loader, device, checkpoint_filename, num_epochs, optimizer, criterion)

In [ ]:
plt.semilogy(history['train_loss'])
plt.plot(history['val_loss'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.ylim(0,1)
plt.show()

plt.plot(history['train_R2'])
plt.plot(history['val_R2'])
plt.xlabel('Epoch')
plt.ylabel('R$^2$')
plt.ylim(0,1.0)
plt.show()

In [ ]:
from sklearn.metrics import r2_score

# 1. Define your model configuration (must match what you used during training).
model_config = {
    # spatiotemporal CNN encoder 
    'input_channels':1,
    'cnn_output_shape':1024,
    'num_cnn_layers':5,
    'blocks_per_stage':2,
    'start_filters':32,
    'input_frames':10,
    
    # mlp parameter encoder
    'param_input_shape':2,
    'param_hidden_layers':[8,8],
    'param_output_shape':8,
    
    # mlp prediction head
    'hidden_layers':[256,32],
    'output_shape':7
}

# 2. Create the model instance.
model = models.PropertyPredictor(model_config)

# 3. Load the saved weights. 
#    Use map_location if you're loading on a different device than training (e.g., CPU).
checkpoint_path = checkpoint_filename
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.load_state_dict(torch.load(checkpoint_path, map_location=device))

# 4. Move the model to the appropriate device (CPU or GPU).
model.to(device)

# 5. Set the model to evaluation mode.
model.eval()

def evaluate_regression(loader, model, device):
    """
    Evaluate the model over a DataLoader and return all targets and predictions.
    
    Args:
        loader: DataLoader yielding (inputs, (targets, params)) batches.
        model: Trained model.
        device: torch.device to run inference on.
    
    Returns:
        all_targets: numpy array of shape (N, output_dim)
        all_preds: numpy array of shape (N, output_dim)
    """
    model.eval()
    all_targets = []
    all_preds = []
    with torch.no_grad():
        for inputs, (targets, params) in loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            params = params.to(device)
            
            outputs = model(inputs, params)
            
            # Append batch predictions and targets.
            all_targets.append(targets.cpu().numpy())
            all_preds.append(outputs.cpu().numpy())
    all_targets = np.concatenate(all_targets, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)
    return all_targets, all_preds

# Evaluate the training and validation sets.
train_targets, train_preds = evaluate_regression(train_loader, model, device)
val_targets, val_preds = evaluate_regression(val_loader, model, device)

# Compute overall R² scores for each dataset by flattening the arrays.
overall_r2_train = r2_score(train_targets.flatten(), train_preds.flatten())
overall_r2_val   = r2_score(val_targets.flatten(), val_preds.flatten())

# Compute per-sample R² scores (each sample is a 7-dimensional vector).
per_sample_r2_train = [r2_score(train_targets[i], train_preds[i]) for i in range(train_targets.shape[0])]
per_sample_r2_val   = [r2_score(val_targets[i], val_preds[i]) for i in range(val_targets.shape[0])]

print("Overall training R²:", overall_r2_train)
print("Overall validation R²:", overall_r2_val)

# Plot predicted vs. true for one output dimension (e.g. first element of the regression vector)
plt.figure(figsize=(12,5))

# Training scatter plot
plt.subplot(1,2,1)
plt.scatter(train_targets[:,0], train_preds[:,0], alpha=0.5)
plt.xlabel("True (Dimension 0)")
plt.ylabel("Predicted (Dimension 0)")
plt.title("Training: Predicted vs True (Dimension 0)")
plt.text(0.05, 0.95, f"R²: {overall_r2_train*100:.2f}%", transform=plt.gca().transAxes,
         fontsize=12, verticalalignment='top', bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

In [ ]:
# List of property names in order.
property_names = [
    'cp_metal', 
    'h_vapor', 
    'kappa_metal', 
    'laser_reflectivity', 
    'mass_density_metal', 
    't_boil', 
    'tcrit'
]

def plot_property_predictions(true_array, pred_array, dataset_name, filename):
    """
    Plots true vs. predicted values for each regression target.
    
    Args:
        true_array (np.ndarray): Array of true values of shape (N, 7).
        pred_array (np.ndarray): Array of predicted values of shape (N, 7).
        dataset_name (str): Name to display on the title (e.g., 'Training' or 'Validation').
        filename (str): Filename to save the figure (e.g., "train_property_predictions.png").
    """
    n_properties = len(property_names)
    
    # Create a grid of subplots. Here we use 4 rows and 2 columns (8 slots) and leave one slot empty.
    fig, axes = plt.subplots(4, 2, figsize=(12, 16))
    axes = axes.flatten()
    
    for i in range(n_properties):
        ax = axes[i]
        true_vals = true_array[:, i]
        pred_vals = pred_array[:, i]
        r2 = r2_score(true_vals, pred_vals)
        
        ax.scatter(true_vals, pred_vals, alpha=0.25)
        ax.set_xlabel("True " + property_names[i])
        ax.set_ylabel("Predicted " + property_names[i])
        ax.set_title(f"{dataset_name} - {property_names[i]}")
        # Place the R^2 value in the upper left corner of the subplot.
        ax.text(0.05, 0.95, f"R²: {r2:.2f}", transform=ax.transAxes,
                fontsize=10, verticalalignment='top', bbox=dict(boxstyle="round", facecolor="white", alpha=0.5))
    
    # Remove any unused subplot (if grid has extra slot(s))
    for j in range(n_properties, len(axes)):
        fig.delaxes(axes[j])
    
    fig.tight_layout()
    # Save the figure with transparency and high dpi.
    fig.savefig(filename, transparent=True, dpi=600)
    plt.show()

# Example usage:
# Assuming train_targets, train_preds, val_targets, and val_preds have been computed.
# They should be numpy arrays of shape (N, 7)
plot_property_predictions(train_targets, train_preds, "Training", "Group1a-b train_property_predictions.png")
plot_property_predictions(val_targets, val_preds, "Validation", "Group1a-b val_property_predictions.png")